In [ ]:
import numpy as np
import cvxpy as cp
from itertools import product
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

In [ ]:
# Linear system matrices.
h = .1
A = np.array([[1, h], [0, 1]])
B = np.array([0, h])

# Time horizon.
K = 8

# Initial state.
x0 = np.array([1, 10])

# Terminal constraint.
xK_max = np.array([1, 1])
xK_min = - xK_max

# Vertices of disturbance set.
W = np.array([[1, -1], [-1, -1], [0, 1]]) * .1

In [ ]:
# Variables.
X = cp.Variable((K, 2)) # matrix with the state at each time step
U = cp.Variable(K - 1) # vector with the contro input at each time step

# Initial conditions.
constraints = [X[0] == x0]

# Discrete-time dynamics.
for k in range(K - 1):
    constraints.append(X[k + 1] == A @ X[k] + B * U[k])

# Robust terminal constraint.
CK = np.hstack([A ** (K - 1 - k) for k in range(K)])
for wK in product(W, repeat=K):
    xK_tilde = X[-1] + CK @ np.concatenate(wK)
    constraints.append(xK_tilde >= xK_min)
    constraints.append(xK_tilde <= xK_max)

# Objective function.
obj = h * cp.sum_squares(U)

# Print number of constraints.
print('Problem has', len(constraints), 'constraints')

# Solve problem.
prob = cp.Problem(cp.Minimize(obj), constraints)
prob.solve()

In [ ]:
# Plot terminal set.
plt.figure()
d = xK_max - xK_min
rect = Rectangle(xK_min, *d, ec='b', fc='none', label='Terminal set')
plt.gca().add_patch(rect)

# Plot state trajectory with random disturbances.
np.random.seed(0)
n_trials = 100 # number of sampled disturbance sequences
X_real = np.zeros((K, 2))
X_real[0] = x0
for i in range(n_trials):
    for k in range(K - 1):
        theta = np.random.rand(len(W))
        wk = theta @ W / sum(theta) # sample disturbance
        X_real[k + 1] = A @ X_real[k] + B * U[k].value + wk
    label ='Sampled trajectory' if i == 0 else None
    plt.plot(*X_real.T, 'r:', label=label)

# Plot nominal state trajectory.
plt.plot(*X.value.T, 'g', label='Nominal trajectory')

# Plot options.
plt.xlabel(r'$x_1$')
plt.ylabel(r'$x_2$')
plt.title('State trajectory')
plt.legend()
plt.grid()